# Controlled response estimation: meaningful labels

Same measurement as notebook 02 for ten opinion sets with real options (q = 2 to 10), 49 displayed agents, seven-point grid.  Writes `data/raw/semantic_queries.csv`.

In [ ]:
import os

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # before importing vllm, see README

import json
import random
import re
import string
import time
from pathlib import Path

import pandas as pd
from vllm import LLM, SamplingParams

In [ ]:
SEED = 7

MODELS = {
    "llama3_70b_awq": ("TechxGenus/Meta-Llama-3-70B-Instruct-AWQ",
                       dict(quantization="awq", gpu_memory_utilization=0.90, max_model_len=2048, max_num_seqs=20)),
    "llama31_8b_it": ("meta-llama/Llama-3.1-8B-Instruct",
                      dict(gpu_memory_utilization=0.85, max_model_len=2048, max_num_seqs=32)),
    "qwen25_7b_it": ("Qwen/Qwen2.5-7B-Instruct",
                     dict(gpu_memory_utilization=0.85, max_model_len=2048, max_num_seqs=32)),
    "qwen25_32b_it": ("Qwen/Qwen2.5-32B-Instruct",
                      dict(gpu_memory_utilization=0.90, max_model_len=2048, max_num_seqs=8)),
    "gemma4_E4B_it": ("google/gemma-4-E4B-it",
                      dict(gpu_memory_utilization=0.70, max_model_len=2048, max_num_seqs=8)),
    "gemma4_31B_dense": ("google/gemma-4-31B-it",
                         dict(gpu_memory_utilization=0.82, max_model_len=2048, max_num_seqs=4)),
}
ACTIVE = ["gemma4_E4B_it", "gemma4_31B_dense"]

# opinion sets and queries per state (more queries for larger sets, so every option leads often enough)
SETS = [
    ("energy_q2_anchor", ["renewable energy", "fossil fuels"], 150),
    ("energy_q3", ["renewable energy", "nuclear power", "fossil fuels"], 150),
    ("justice_q3", ["rehabilitative justice", "punitive justice", "restorative justice"], 150),
    ("speech_q3", ["unrestricted free speech", "regulated speech", "harm-reduction moderation"], 150),
    ("ai_governance_q4", ["open-source AI", "corporate self-regulation", "government regulation",
                          "international AI treaty"], 150),
    ("beverage_q5", ["coffee", "tea", "water", "juice", "soda"], 150),
    ("political_ideology_q6", ["liberalism", "conservatism", "socialism", "libertarianism", "green politics",
                               "nationalism"], 240),
    ("pets_q7", ["cats", "dogs", "birds", "fish", "rabbits", "hamsters", "reptiles"], 280),
    ("policy_priorities_q10", ["healthcare", "education", "climate policy", "economic growth", "public safety",
                               "housing", "immigration", "digital rights", "national defense", "social welfare"], 300),
    ("ai_values_q10", ["helpfulness", "honesty", "harmlessness", "privacy", "fairness", "transparency",
                       "autonomy", "robustness", "accountability", "efficiency"], 300),
]
N_DISPLAY = 49
REL_GRID = [0.0, 0.1, 0.25, 0.4, 0.6, 0.8, 1.0]

BATCH_SIZE = 64
TEMPERATURE = 0.2
TOP_P = 0.9
MAX_NEW_TOKENS = 24
NAME_LENGTH = 3

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## Prompt and parser

In [ ]:
def leader_counts(q, n_display, rel):
    c_min = n_display // q + (1 if n_display % q else 0)
    c_max = n_display - (q - 1)
    leader = round(c_min + rel * (c_max - c_min))
    rest = n_display - leader
    return [leader] + [rest // (q - 1) + (1 if i < rest % (q - 1) else 0) for i in range(q - 1)]


def random_names(n, rng):
    chars = string.ascii_letters + string.digits
    names = set()
    while len(names) < n:
        names.add("".join(rng.choices(chars, k=NAME_LENGTH)))
    return list(names)


def create_prompt(names, opinions):
    lines = ["Below you can see the list of all the other AI agents with the opinion they support.",
             "You must reply with the opinion you want to support.",
             "The opinion must be reported between square brackets.", ""]
    lines += [f"{n}: {o}" for n, o in zip(names, opinions)]
    lines.append("Reply only with the opinion you want to support, between square brackets.")
    return "\n".join(lines)


def parse_reply(text, labels):
    found = re.findall(r"\[([^\]]+)\]", text)
    lowered = {l.lower(): l for l in labels}
    for c in found if found else [text]:
        c = c.strip().strip(" .,:;!?'\"").lower()
        if c in lowered:
            return lowered[c]
    return None


def build_query(counts, options, leader, rng):
    # option `leader` takes role 0, the others keep their order
    display = [options[leader]] + [o for i, o in enumerate(options) if i != leader]
    opinions = []
    for role, c in enumerate(counts):
        opinions += [display[role]] * c
    names = random_names(len(opinions), rng)
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    prompt = create_prompt([n for n, _ in pairs], [o for _, o in pairs])
    return prompt, {d: r for r, d in enumerate(display)}


def chat_format(llm, prompts):
    tok = llm.get_tokenizer()
    return [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

## Run

In [ ]:
def run_state(llm, model_label, set_name, options, rel, counts, n_queries, seed):
    rng = random.Random(seed)
    q = len(options)
    queries = [build_query(counts, options, i % q, rng) for i in range(n_queries)]     # leader rotates
    rows = []
    for start in range(0, n_queries, BATCH_SIZE):
        batch = queries[start:start + BATCH_SIZE]
        outputs = llm.generate(chat_format(llm, [b[0] for b in batch]), sampling, use_tqdm=False)
        for k, ((_, display_to_role), out) in enumerate(zip(batch, outputs)):
            chosen = parse_reply(out.outputs[0].text, list(display_to_role))
            rows.append({"model_label": model_label, "set_name": set_name, "q": q, "n_display": N_DISPLAY,
                         "rel": rel, "leader_option_idx": (start + k) % q,
                         "shares_options": json.dumps([counts[display_to_role[o]] / N_DISPLAY for o in options]),
                         "chosen_option_idx": options.index(chosen) if chosen else None,
                         "chosen_role": display_to_role.get(chosen) if chosen else None,
                         "reply": out.outputs[0].text})
    return pd.DataFrame(rows)


sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)
out_file = RAW / "semantic_queries.csv"
cell = 0
for model_label in ACTIVE:
    model_id, kwargs = MODELS[model_label]
    llm = LLM(model=model_id, trust_remote_code=True, tensor_parallel_size=1, enforce_eager=True,
              disable_log_stats=True, **kwargs)
    for set_name, options, n_queries in SETS:
        for rel in REL_GRID:
            cell += 1
            counts = leader_counts(len(options), N_DISPLAY, rel)
            t0 = time.time()
            df = run_state(llm, model_label, set_name, options, rel, counts, n_queries, seed=SEED + cell)
            df.to_csv(out_file, mode="a", header=not out_file.exists(), index=False)
            print(f"{model_label} {set_name} rel={rel:.2f}  valid={df['chosen_role'].notna().mean():.3f}  "
                  f"P(leader)={(df['chosen_role'] == 0).mean():.3f}  {time.time() - t0:.0f}s")
    del llm